[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/use_cases/regulatory_intelligence/notebook/regulatory_intelligence.ipynb)

# Regulatory Intelligence

An end-to-end Semantica pipeline that turns real U.S. federal AI-governance and cybersecurity regulations into an explainable, ontology-driven knowledge graph.

## Use case

- Federal AI-governance and cybersecurity regulations are published independently by different agencies (NIST, OMB, HHS, the Federal Reserve) with no cross-referencing between documents.
- A compliance question that spans several of them, such as "which regulations apply to an AI system in sector X," "do these two frameworks agree," or "what changed between versions," currently requires a human to read all of them and cross-reference manually.
- This notebook builds a knowledge graph that answers those questions directly, with cited evidence, computed (not narrated) conflict and diff detection, and policy-gated, auditable decisions for two sectors (healthcare, financial services).
- Scope is deliberately narrow: 9 real documents, not full corpora. See "Scope" at the end.

> [!NOTE]
> Every document and ontology used here is real and publicly sourced. None of it is synthetic or LLM-generated. See `../data/README.md` and `../ontology/README.md` for exact source URLs and retrieval dates. The two hand-authored files (`regulatory_extension.ttl`, `regulatory_taxonomy.ttl`) are schema, not data. Every term in them was verified against the real source text before being written.

## Pipeline

```
 Real Documents (PDF / XML)
        │
        ▼
 Ingestion            PDFParser · DoclingParser · ingest_xml
        │
        ▼
 Chunking             TextSplitter
        │
        ▼
 Extraction           NERExtractor · RelationExtractor · TripletExtractor
        │
        ▼
 Ontology Import      OntologyIngestor  ◄──── 6 real W3C/SPAR ontologies
        │                                     (ORG · PROV-O · SKOS · DCAT · OWL-Time · FRBR)
        ▼
 Curated Requirement Clauses     JSONParser
        │
        ▼
 Entity Resolution    EntityResolver · SimilarityCalculator
        │
        ▼
 Knowledge Graph      ContextGraph via GraphBuilder
        │
        ├──► Ontology Generation & Evaluation   OntologyGenerator · OntologyEvaluator
        ├──► SHACL Validation                   SHACLGenerator · pyshacl
        ├──► Deterministic Reasoning             Reasoner (forward-chaining)
        ├──► Provenance                          ProvenanceManager (PROV-O)
        └──► Persistent RDF Database             Oxigraph (on-disk) + TripletStore (Blazegraph/Jena)
        │
        ▼
 Conflict Detection · Temporal Reasoning    ConflictDetector · TemporalVersionManager
        │
        ▼
 SPARQL · JSON-LD                            Oxigraph · rdflib · RDFExporter
        │
        ▼
 GraphRAG Retrieval                          AgentContext.query_with_reasoning()
        │
        ▼
 Decision Intelligence      PolicyEngine · CausalChainAnalyzer · precedent search · audit report
        │
        ▼
 Explainable, evidence-backed answer
```

## What each layer demonstrates

- **Ingestion**: `PDFParser` (fast) and `DoclingParser` (layout-aware, used selectively) turn heterogeneous file formats into normalized text.
- **Chunking**: `TextSplitter` breaks every document into bounded, citation-addressable units.
- **Extraction**: `NERExtractor`, `RelationExtractor`, and `TripletExtractor` run automatic entity, relation, and triplet extraction across all 9 documents; used to show why this pipeline also relies on curated data for dense legal text.
- **Ontology**: `OntologyIngestor` reuses 6 real external ontologies rather than inventing new ones. `OntologyGenerator` and `OntologyEvaluator` generate and score a working ontology from the graph itself.
- **Validation**: `SHACLGenerator` and `pyshacl` validate instance data against structural constraints.
- **Reasoning**: `Reasoner` performs deterministic, rule-based forward-chaining inference, distinct from the LLM-based reasoning used later in GraphRAG.
- **Provenance**: `ProvenanceManager` emits real W3C PROV-O lineage for every fact.
- **Storage**: an Oxigraph store gives genuine on-disk RDF persistence with zero extra infrastructure; `TripletStore` is Semantica's own interface to a dedicated production graph-database server (Blazegraph, Jena, RDF4J, AnzoGraph).
- **Cross-document reasoning**: `ConflictDetector` and `TemporalVersionManager` find real disagreements and diffs between frameworks.
- **Retrieval**: `AgentContext.query_with_reasoning()` implements GraphRAG, retrieval that expands across graph edges rather than text similarity alone.
- **Decision Intelligence**: `PolicyEngine`, `CausalChainAnalyzer`, precedent search, and a decision audit report treat AI-assisted decisions as first-class, queryable, explainable graph objects.

## Questions this notebook answers

- Which cybersecurity regulations apply to hospitals? See Step 18 (GraphRAG).
- Which policies contradict each other? See Step 14 (Conflict Detection).
- What changed between framework versions? See Step 15 (Temporal Reasoning).
- Show every regulation related to AI transparency. See Step 16 (SPARQL).
- Can Hospital X or Bank Y deploy this AI system under current regulations? See Step 19 (Decision Intelligence).

## Install

```bash
pip install semantica[shacl] pdfplumber rdflib requests pyoxigraph
# Optional, for higher-fidelity PDF parsing in Step 1:
pip install semantica[parse-docling]
```


In [1]:
import sys, os, json
sys.path.insert(0, os.path.abspath("../../../../"))

BASE = os.path.abspath("..")
DATA_DIR = os.path.join(BASE, "data")
DATA_RAW = os.path.join(DATA_DIR, "raw")
ONTOLOGY_EXTERNAL = os.path.join(BASE, "ontology", "external")
ONTOLOGY_DIR = os.path.join(BASE, "ontology")

print("Data dir:", DATA_RAW)
print("Ontology dir:", ONTOLOGY_DIR)
assert os.path.isdir(DATA_RAW), "Run data/download_data.py first"
assert os.path.isdir(ONTOLOGY_EXTERNAL), "Run ontology/download_ontologies.py first"


Data dir: C:\Users\moham\semantica\cookbook\use_cases\regulatory_intelligence\data\raw
Ontology dir: C:\Users\moham\semantica\cookbook\use_cases\regulatory_intelligence\ontology


---
# Part A. Ingestion, Extraction, and Graph Construction

## Step 1. Ingest the 9 real documents

- 8 documents use `PDFParser` (fast, `pdfplumber`-based, flattens to plain text).
- The Federal Reserve compliance plan uses `DoclingParser` instead, a layout-aware, ML-based converter that preserves headings and tables as Markdown. It costs tens of seconds per document rather than sub-second, which is why it's used on one document rather than all nine: an explicit accuracy and speed tradeoff, not an oversight.
- If `docling` isn't installed, this cell falls back to `PDFParser` automatically.
- The HIPAA Security Rule is ingested as XML via `ingest_xml`, not PDF (see `../data/README.md` for why).


In [2]:
# TORCHDYNAMO_DISABLE avoids a torch.compile/C++-toolchain requirement that
# Docling's layout model can trigger on machines without a C++ compiler installed.
os.environ.setdefault("TORCHDYNAMO_DISABLE", "1")

from semantica.parse import PDFParser
from semantica.ingest.methods import ingest_xml

pdf_parser = PDFParser()

try:
    from semantica.parse.docling_parser import DoclingParser, DOCLING_AVAILABLE
    docling_parser = DoclingParser(export_format="markdown") if DOCLING_AVAILABLE else None
except ImportError:
    docling_parser = None

PDF_DOCS = {
    "nist_ai_rmf_1.0": "nist_ai_rmf_1.0.pdf",
    "nist_csf_1.1": "nist_csf_1.1.pdf",
    "nist_csf_2.0": "nist_csf_2.0.pdf",
    "nist_sp800-66r2": "nist_sp800-66r2_hipaa_security.pdf",
    "eo_14110": "eo_14110_safe_secure_trustworthy_ai.pdf",
    "omb_m24-10": "omb_m24-10_ai_governance.pdf",
    "nist_ai_600-1": "nist_ai_600-1_genai_profile.pdf",
}
DOCLING_DOC = ("fed_compliance_m24-10", "fed_compliance_plan_omb_m24-10.pdf")

document_text = {}
document_pages = {}
parser_used = {}

for doc_id, filename in PDF_DOCS.items():
    result = pdf_parser.parse(os.path.join(DATA_RAW, filename))
    document_text[doc_id] = result["full_text"]
    document_pages[doc_id] = result["total_pages"]
    parser_used[doc_id] = "PDFParser"
    print(f"  [PDFParser]  {doc_id:28s} {result['total_pages']:4d} pages  {len(result['full_text']):8,d} chars")

doc_id, filename = DOCLING_DOC
if docling_parser is not None:
    try:
        result = docling_parser.parse(os.path.join(DATA_RAW, filename))
        parser_used[doc_id] = "DoclingParser"
    except Exception as exc:
        print(f"  Docling failed ({exc}); falling back to PDFParser")
        result = pdf_parser.parse(os.path.join(DATA_RAW, filename))
        parser_used[doc_id] = "PDFParser (fallback)"
else:
    result = pdf_parser.parse(os.path.join(DATA_RAW, filename))
    parser_used[doc_id] = "PDFParser (Docling not installed)"
document_text[doc_id] = result["full_text"]
document_pages[doc_id] = result["total_pages"]
print(f"  [{parser_used[doc_id]:22s}] {doc_id:28s} {result['total_pages']:4d} pages  {len(result['full_text']):8,d} chars")
if parser_used[doc_id] == "DoclingParser":
    heading_lines = [l for l in result["full_text"].splitlines() if l.strip().startswith("#")]
    print(f"    Markdown structure preserved: {len(heading_lines)} headings detected, e.g. {heading_lines[:2]}")

# HIPAA Security Rule: real eCFR XML, fetched via eCFR's public versioner API
xml_data = ingest_xml(os.path.join(DATA_RAW, "hipaa_security_rule_45cfr164_subpart_c.xml"))
document_text["hipaa_45cfr164_subpart_c"] = " ".join(
    el["text"] for el in xml_data.elements if el.get("text")
)
parser_used["hipaa_45cfr164_subpart_c"] = "ingest_xml"
print(f"  [ingest_xml]           {'hipaa_45cfr164_subpart_c':28s} {'':4s}  {len(document_text['hipaa_45cfr164_subpart_c']):8,d} chars")

print(f"\nIngested {len(document_text)} documents using {len(set(parser_used.values()))} different parsers.")


Status,Action,Module,Submodule,Progress,ETA,Rate,Time,Extracted
🔄,Semantica is storing,🗄️ triplet_store,QueryEngine,-,-,-,0.96s,-
❌,Semantica is storing,🗄️ triplet_store,BlazegraphStore,-,-,-,0.00s,-
✅,Semantica is exporting,💾 export,RDFExporter,100.0%,-,-,0.01s,-
✅,Semantica is processing,🔗 context,AgentMemory,100.0%,-,-,0.03s,-
❌,Semantica is embedding,💾 embeddings,TextEmbedder,-,-,-,0.00s,-
✅,Semantica is indexing,📊 vector_store,FAISSStore,100.0%,-,-,0.00s,-
✅,Semantica is processing,🔗 context,ContextRetriever,100.0%,-,-,0.07s,-
❌,Semantica is indexing,📊 vector_store,HybridSearch,-,-,-,0.00s,-
✅,Semantica is building,🧠 kg,CentralityCalculator,100.0%,-,-,0.00s,-
✅,Semantica is building,🧠 kg,CommunityDetector,100.0%,-,-,0.01s,-


🔄 Semantica is parsing: PDF: nist_ai_rmf_1.0.pdf 🔍 parse PDFParser |░░░░░░░░░░░░░░░| 0.0% ETA: - Rate: - Time: 0.00s Extracted: -

🔄 Semantica is parsing: Parsing 48 pages 🔍 parse PDFParser |░░░░░░░░░░░░░░░| 0.0% ETA: - Rate: - Time: 0.17s Extracted: -

✅ Semantica is parsing: Parsed 48 pages 🔍 parse PDFParser |███████████████| 100.0% ETA: - Rate: - Time: 3.42s Extracted: -

  [PDFParser]  nist_ai_rmf_1.0                48 pages   101,280 chars


...

🔄 Semantica is parsing: Extracting tables from document... 🔍 parse DoclingParser |█████████████░░| 90.0% ETA: 4.0s Rate: 0.2/s Time: 36.25s Extracted: -

🔄 Semantica is parsing: Extracting page structure... 🔍 parse DoclingParser |█████████████░░| 90.0% ETA: 4.0s Rate: 0.2/s Time: 36.25s Extracted: -

✅ Semantica is parsing: Parsed document (Docling): 10 pages extracted 🔍 parse DoclingParser |███████████████| 100.0% ETA: - Rate: 0.2/s Time: 36.26s Extracted (Docling): 0 tables, 0 images, 10 pages

  [DoclingParser         ] fed_compliance_m24-10          10 pages    17,269 chars
    Markdown structure preserved: 18 headings detected, e.g. ['## AI Governance Bodies', '## Expected Outcomes']


  [ingest_xml]           hipaa_45cfr164_subpart_c             11,599 chars

Ingested 9 documents using 3 different parsers.


## Step 2. Chunk every document

Extraction and LLM based methods have effective context limits, so chunking keeps each unit small enough to process reliably. Chunk boundaries also give provenance something precise to point at: this chunk of this document, not somewhere in a 122 page PDF.

`TextSplitter` supports several strategies (recursive, semantic, structural, sliding window, table aware). `method="recursive"` is a robust general default that requires no model. Applied here to all 9 ingested documents, not a single representative one.


In [3]:
from semantica.split import TextSplitter

splitter = TextSplitter(method="recursive", chunk_size=1500, chunk_overlap=150)

all_chunks = {}
for doc_id, text in document_text.items():
    all_chunks[doc_id] = splitter.split(text)

total_chunks = sum(len(v) for v in all_chunks.values())
print(f"Chunked all {len(all_chunks)} documents into {total_chunks} chunks total:")
for doc_id, doc_chunks in all_chunks.items():
    print(f"  {doc_id:28s} {len(document_text[doc_id]):8,d} chars -> {len(doc_chunks):4d} chunks")


Chunked all 9 documents into 840 chunks total:
  nist_ai_rmf_1.0               101,280 chars ->   84 chunks
  nist_csf_1.1                  125,406 chars ->  104 chunks
  nist_csf_2.0                   68,315 chars ->   55 chunks
  nist_sp800-66r2               299,487 chars ->  246 chunks
  eo_14110                      143,549 chars ->  113 chunks
  omb_m24-10                    101,817 chars ->   82 chunks
  nist_ai_600-1                 158,280 chars ->  130 chunks
  fed_compliance_m24-10          17,269 chars ->   16 chunks
  hipaa_45cfr164_subpart_c       11,599 chars ->   10 chunks


## Step 3. Extract entities, relations, and triplets across the corpus

`NERExtractor(method="pattern")`, `RelationExtractor(method="pattern")`, and `TripletExtractor(method="pattern")` run fully automatically, with no model download and no API key required.

Applied here to the first 3 chunks of every one of the 9 documents (27 chunks total, not a single sample) to give a representative, corpus wide picture rather than one lucky or unlucky excerpt. The result makes a concrete point: pattern based extraction over dense regulatory prose is noisy. Institution names get mislabeled, and most sentences match no relation pattern at all.

This is why the rest of this notebook uses `data/requirement_clauses.json`, 20 requirement clauses hand curated from the real text with a verified citation each, as the authoritative dataset rather than trusting fully automatic extraction over legal and regulatory language. The automatic path shown here is real and available. It is a precision and recall tradeoff, not a missing feature.


In [4]:
from semantica.semantic_extract import NERExtractor, RelationExtractor, TripletExtractor

ner = NERExtractor(method="pattern")
rel = RelationExtractor(method="pattern")
te = TripletExtractor(method="pattern")

CHUNKS_PER_DOC = 3
corpus_entities, corpus_relations, corpus_triplets = [], [], []

for doc_id, doc_chunks in all_chunks.items():
    for chunk in doc_chunks[:CHUNKS_PER_DOC]:
        ents = ner.extract_entities(chunk.text)
        rels = rel.extract_relations(chunk.text, ents)
        trs = te.extract_triplets(chunk.text)
        corpus_entities.extend((doc_id, e) for e in ents)
        corpus_relations.extend((doc_id, r) for r in rels)
        corpus_triplets.extend((doc_id, t) for t in trs)

print(f"Across {len(all_chunks)} documents x up to {CHUNKS_PER_DOC} chunks each:")
print(f"  {len(corpus_entities)} entities, {len(corpus_relations)} relations, {len(corpus_triplets)} triplets extracted")

print("\nSample entities:")
for doc_id, e in corpus_entities[:8]:
    print(f"  [{doc_id:20s}] {e.text!r:35s} label={e.label:10s} confidence={e.confidence}")

print("\nSample relations:")
for doc_id, r in corpus_relations[:8]:
    print(f"  [{doc_id:20s}] {r.subject.text} --{r.predicate}--> {r.object.text}  (confidence={r.confidence})")

print("\nSample triplets:")
for doc_id, t in corpus_triplets[:8]:
    print(f"  [{doc_id:20s}] ({t.subject}, {t.predicate}, {t.object})  confidence={t.confidence}")


🔄 Semantica is extracting: Extracting named entities from text 🎯 semantic_extract NERExtractor |░░░░░░░░░░░░░░░| 0.0% ETA: - Rate: - Time: 0.00s Extracted: -

✅ Semantica is extracting: Extracted 2 relations 🎯 semantic_extract RelationExtractor |███████████████| 100.0% ETA: - Rate: - Time: 1.03s Extracted: -

✅ Semantica is extracting: Extracted 2 relations 🎯 semantic_extract RelationExtractor |███████████████| 100.0% ETA: - Rate: - Time: 0.17s Extracted: -

🔄 Semantica is extracting: Starting triplet extraction... 0/1 methods (remaining: 1) 🎯 semantic_extract TripletExtractor |░░░░░░░░░░░░░░░| 0.0% ETA: - Rate: - Time: 0.19s Extracted: -

🔄 Semantica is extracting: Extracting triplets using pattern... (1/1, remaining: 0 methods) 🎯 semantic_extract TripletExtractor |███████████████| 100.0% ETA: - Rate: 5.2/s Time: 0.19s Extracted: -

...

✅ Semantica is extracting: Extracted 10 relations 🎯 semantic_extract RelationExtractor |███████████████| 100.0% ETA: - Rate: - Time: 0.12s Extracted: -

✅ Semantica is extracting: Extracted 10 relations 🎯 semantic_extract RelationExtractor |███████████████| 100.0% ETA: - Rate: - Time: 0.09s Extracted: -

🔄 Semantica is extracting: Starting triplet extraction... 0/1 methods (remaining: 1) 🎯 semantic_extract TripletExtractor |░░░░░░░░░░░░░░░| 0.0% ETA: - Rate: - Time: 0.11s Extracted: -

🔄 Semantica is extracting: Extracting triplets using pattern... (1/1, remaining: 0 methods) 🎯 semantic_extract TripletExtractor |███████████████| 100.0% ETA: - Rate: 8.8/s Time: 0.11s Extracted: -

Across 9 documents x up to 3 chunks each:
  285 entities, 393 relations, 390 triplets extracted

Sample entities:
  [nist_ai_rmf_1.0     ] 'Artificial Intelligence Risk Management\nFramework' label=PERSON     confidence=0.7
  [nist_ai_rmf_1.0     ] 'Artificial Intelligence Risk Management\nFramework' label=PERSON     confidence=0.7
  [nist_ai_rmf_1.0     ] 'National Institute'                label=PERSON     confidence=0.7
  [nist_ai_rmf_1.0     ] 'The\nFrameworkwillemployatwo'      label=PERSON     confidence=0.7
  [nist_ai_rmf_1.0     ] '6028'                              label=DATE       confidence=0.7
  [nist_ai_rmf_1.0     ] '6028'                              label=DATE       confidence=0.7
  [nist_ai_rmf_1.0     ] 'Minor'                             label=UNKNOWN    confidence=0.5
  [nist_ai_rmf_1.0     ] 'Playbook'                          label=UNKNOWN    confidence=0.5

Sample relations:
  [nist_ai_rmf_1.0     ] Artificial Intelligence Risk Management
Framework --related_to--

## Step 4. Import the 6 real external ontologies

- An ontology declares what kinds of things exist (classes, e.g. `org:Organization`) and how they relate (properties, e.g. `prov:wasGeneratedBy`).
- Reused rather than invented: every capability below already has a mature W3C or W3C-affiliated ontology.
- `OntologyIngestor` parses each file with `rdflib` under the hood. Turtle and RDF/XML both load through the same call.


In [5]:
from semantica.ingest import OntologyIngestor

ontology_ingestor = OntologyIngestor()

EXTERNAL_ONTOLOGIES = {
    "org": ("org.ttl", "models agencies as org:Organization"),
    "prov-o": ("prov-o.ttl", "provenance/lineage for every fact"),
    "skos-core": ("skos-core.rdf", "the controlled vocabulary in Step 5"),
    "dcat": ("dcat.ttl", "catalogs each document as a dataset"),
    "time": ("time.ttl", "formal validity intervals"),
    "frbr": ("frbr.ttl", "Work/Expression modeling for Step 15"),
}

imported_ontologies = {}
for name, (filename, purpose) in EXTERNAL_ONTOLOGIES.items():
    ont = ontology_ingestor.ingest_ontology(os.path.join(ONTOLOGY_EXTERNAL, filename))
    imported_ontologies[name] = ont
    print(f"  {name:12s} format={ont.format:8s} classes={len(ont.data.get('classes', [])):4d}  properties={len(ont.data.get('properties', [])):4d}  ({purpose})")

regulatory_extension = ontology_ingestor.ingest_ontology(
    os.path.join(ONTOLOGY_DIR, "regulatory_extension.ttl")
)
regulatory_taxonomy = ontology_ingestor.ingest_ontology(
    os.path.join(ONTOLOGY_DIR, "skos", "regulatory_taxonomy.ttl")
)
print(f"\nHand-authored extension classes: {[c['name'] for c in regulatory_extension.data.get('classes', [])]}"
      f", defined as extensions of the ontologies above, not a parallel schema.")


🔄 Semantica is ingesting: Ontology: org.ttl 📥 ingest OntologyIngestor |░░░░░░░░░░░░░░░| 0.0% ETA: - Rate: - Time: 0.00s Extracted: -

  org          format=turtle   classes=   9  properties=  35  (models agencies as org:Organization)


🔄 Semantica is ingesting: Converting to internal format... 📥 ingest OntologyIngestor |░░░░░░░░░░░░░░░| 0.0% ETA: - Rate: - Time: 0.12s Extracted: -

  prov-o       format=turtle   classes=  51  properties=  69  (provenance/lineage for every fact)


  skos-core    format=xml      classes=   4  properties=  28  (the controlled vocabulary in Step 5)


...

  dcat         format=turtle   classes=   9  properties=  39  (catalogs each document as a dataset)


  time         format=turtle   classes=  20  properties=  58  (formal validity intervals)


🔄 Semantica is ingesting: Ontology: frbr.ttl 📥 ingest OntologyIngestor |░░░░░░░░░░░░░░░| 0.0% ETA: - Rate: - Time: 0.00s Extracted: -

  frbr         format=turtle   classes=  13  properties=  50  (Work/Expression modeling for Step 15)



Hand-authored extension classes: ['Regulation', 'Requirement Clause', 'Agency'], defined as extensions of the ontologies above, not a parallel schema.


## Step 5. Load the SKOS taxonomy

- SKOS is the W3C standard for controlled vocabularies/taxonomies.
- Every concept in `regulatory_taxonomy.ttl` was verified to be a real term from one of the 9 source documents before being added (see the file's own `skos:scopeNote` citations).
- Semantica's built-in SKOS *management* API (`OntologyEngine.list_vocabularies()`, `.list_concepts()`, `.search_concepts()`) is real, but backed by a `TripletStore` that issues SPARQL `SELECT`/`FILTER(CONTAINS(...))` queries against it, so it's demonstrated in Step 13 once a store connection has been attempted. Here, with no live store required yet, the vendored real Turtle is read directly with `rdflib`. `NamespaceManager.get_skos_uri()` builds the SKOS namespace URI rather than hardcoding the string.


In [6]:
import rdflib
from semantica.ontology import NamespaceManager

REGV = rdflib.Namespace("https://semantica.dev/cookbook/regulatory-intelligence/vocabulary#")
namespace_manager = NamespaceManager()
SKOS = rdflib.Namespace(namespace_manager.get_skos_uri("").rstrip("#") + "#")
REG_BASE = "https://semantica.dev/cookbook/regulatory-intelligence/ontology#"
REG = rdflib.Namespace(REG_BASE)

skos_graph = rdflib.Graph()
skos_graph.parse(os.path.join(ONTOLOGY_DIR, "skos", "regulatory_taxonomy.ttl"), format="turtle")

concepts = {}
for concept_uri in skos_graph.subjects(rdflib.RDF.type, SKOS.Concept):
    pref_label = skos_graph.value(concept_uri, SKOS.prefLabel)
    concepts[str(pref_label)] = str(concept_uri)

print(f"{len(concepts)} real SKOS concepts loaded, e.g.:")
for label in ["Govern", "Administrative Safeguards", "Transparency", "Confabulation", "Healthcare", "Financial Services"]:
    print(f"  {label!r:32s} -> {concepts.get(label)}")


23 real SKOS concepts loaded, e.g.:
  'Govern'                         -> https://semantica.dev/cookbook/regulatory-intelligence/vocabulary#Govern
  'Administrative Safeguards'      -> https://semantica.dev/cookbook/regulatory-intelligence/vocabulary#AdministrativeSafeguards
  'Transparency'                   -> https://semantica.dev/cookbook/regulatory-intelligence/vocabulary#Transparency
  'Confabulation'                  -> https://semantica.dev/cookbook/regulatory-intelligence/vocabulary#Confabulation
  'Healthcare'                     -> https://semantica.dev/cookbook/regulatory-intelligence/vocabulary#Healthcare
  'Financial Services'             -> https://semantica.dev/cookbook/regulatory-intelligence/vocabulary#FinancialServices


## Step 6. Load curated requirement clauses

- `data/requirement_clauses.json` holds 20 requirement clauses, each with `doc`, `sector`, `topic` (a real SKOS concept), `citation`, and `text` (a real substring of the ingested text).
- Loaded via Semantica's own `JSONParser` rather than an inline Python literal.
- Every clause's `text` is re-verified against `document_text` before being trusted, closing the loop with Step 1's raw ingestion.


In [7]:
from semantica.parse import JSONParser

json_parser = JSONParser()
clauses_data = json_parser.parse(os.path.join(DATA_DIR, "requirement_clauses.json"))
REQUIREMENT_CLAUSES = clauses_data.data["requirement_clauses"]

for clause in REQUIREMENT_CLAUSES:
    haystack = document_text[clause["doc"]]
    assert clause["text"].lower() in haystack.lower(), f"NOT FOUND in real text: {clause['id']}"
print(f"Loaded and verified {len(REQUIREMENT_CLAUSES)} requirement clauses.")


🔄 Semantica is parsing: JSON: requirement_clauses.json 🔍 parse JSONParser |░░░░░░░░░░░░░░░| 0.0% ETA: - Rate: - Time: 0.00s Extracted: -

Loaded and verified 20 requirement clauses.


## Step 7. Entity resolution

- The same agency is named inconsistently across independently-written documents ("HHS" vs. "U.S. Department of Health and Human Services"); left unresolved, a graph treats these as two different organizations.
- `EntityResolver.resolve_entities()` is Semantica's batch merge API. The lower-level `SimilarityCalculator` is also called directly to show the real pairwise scores behind that decision, including where the batch merge doesn't actually fire in the installed library version (see `../README.md`, "Notes on real-world library behavior").


In [8]:
from semantica.kg import EntityResolver
from semantica.deduplication.similarity_calculator import SimilarityCalculator

raw_agency_mentions = [
    {"id": "a1", "name": "HHS", "type": "Agency"},
    {"id": "a2", "name": "U.S. Department of Health and Human Services", "type": "Agency"},
    {"id": "a3", "name": "Department of Health and Human Services", "type": "Agency"},
    {"id": "a4", "name": "NIST", "type": "Agency"},
    {"id": "a5", "name": "National Institute of Standards and Technology", "type": "Agency"},
    {"id": "a6", "name": "OMB", "type": "Agency"},
    {"id": "a7", "name": "Office of Management and Budget", "type": "Agency"},
    {"id": "a8", "name": "Federal Reserve", "type": "Agency"},
    {"id": "a9", "name": "Board of Governors of the Federal Reserve System", "type": "Agency"},
]

resolver = EntityResolver(strategy="fuzzy", similarity_threshold=0.6)
resolved_agencies = resolver.resolve_entities(raw_agency_mentions)
print(f"EntityResolver.resolve_entities(): {len(raw_agency_mentions)} raw mentions -> {len(resolved_agencies)} entities")

calc = SimilarityCalculator()
print("\nPairwise similarity for known-duplicate agency name variants:")
for i, j in [(1, 2), (0, 1), (3, 4), (7, 8)]:
    e1, e2 = raw_agency_mentions[i], raw_agency_mentions[j]
    sim = calc.calculate_similarity(e1, e2)
    print(f"  {e1['name']!r:50s} vs {e2['name']!r:55s} -> score={sim.score:.2f}")


🔄 Semantica is building: Resolving entities 🧠 kg EntityResolver |░░░░░░░░░░░░░░░| 0.0% ETA: - Rate: - Time: 0.00s Extracted: -

🔄 Semantica is deduplicating: Comparing candidates... 1/2 🔄 deduplication SimilarityCalculator |███████░░░░░░░░| 50.0% ETA: - Time: 0.00s Extracted: -

🔄 Semantica is deduplicating: Comparing candidates... 2/2 🔄 deduplication SimilarityCalculator |███████████████| 100.0% ETA: - Rate: 217.8/s Time: 0.01s Extracted: -

🔄 Semantica is deduplicating: Creating duplicate candidates... 0/2 (remaining: 2) 🔄 deduplication DuplicateDetector |░░░░░░░░░░░░░░░| 0.0% ETA: - Rate: - Time: 0.02s Extracted: -

🔄 Semantica is deduplicating: Creating duplicate candidates... 1/2 (remaining: 1) 🔄 deduplication DuplicateDetector |███████░░░░░░░░| 50.0% ETA: 0.0s Rate: 50.4/s Time: 0.02s Extracted: -

...

  'U.S. Department of Health and Human Services'     vs 'Department of Health and Human Services'               -> score=0.80


  'HHS'                                              vs 'U.S. Department of Health and Human Services'          -> score=0.54


🔄 Semantica is deduplicating: Calculating string similarity... 🔄 deduplication SimilarityCalculator |░░░░░░░░░░░░░░░| 0.0% ETA: - Rate: - Time: 0.01s Extracted: -

  'NIST'                                             vs 'National Institute of Standards and Technology'        -> score=0.67


  'Federal Reserve'                                  vs 'Board of Governors of the Federal Reserve System'      -> score=0.68


## Step 8. Assemble the knowledge graph

- The graph is described declaratively as two lists, `entities` and `relationships`, and handed to `GraphBuilder`, which populates the `ContextGraph` (via `graph_store=`) rather than calling `add_node()`/`add_edge()` in a manual loop.
- Node types (`Agency`, `Regulation`, `RequirementClause`) map conceptually to `org:Organization` / `dcat:Dataset` / `prov:Entity` from the vendored ontologies (see `regulatory_extension.ttl`).
- Sector/topic edges point at the real SKOS concepts loaded in Step 5. Cross-regulation edges (`supersedes`, `amends`, `implements`) are grounded in the documents themselves, not inferred.
- Real `skos:broader` edges (`Rights-Impacting AI`/`Safety-Impacting AI` pointing to `Risk Classification`, extracted from `regulatory_taxonomy.ttl` itself, not hand-typed) are included as `skos:broader`-typed edges. `ContextGraph` runs `validate_skos_hierarchy()` automatically whenever an edge is typed `skos:broader`/`skos:narrower`, so this exercises Semantica's built-in SKOS cycle-detection for real, demonstrated explicitly in the next cell.


In [9]:
from semantica.context import ContextGraph
from semantica.kg import GraphBuilder

AGENCY_BY_DOC = {
    "nist_ai_rmf_1.0": "NIST", "nist_csf_1.1": "NIST", "nist_csf_2.0": "NIST",
    "nist_sp800-66r2": "NIST", "nist_ai_600-1": "NIST",
    "hipaa_45cfr164_subpart_c": "HHS", "eo_14110": "White House",
    "omb_m24-10": "OMB", "fed_compliance_m24-10": "Federal Reserve",
}

entities = [
    {"id": f"agency:{agency}", "type": "Agency", "name": agency, "properties": {}}
    for agency in sorted(set(AGENCY_BY_DOC.values()))
] + [
    {"id": f"reg:{doc_id}", "type": "Regulation", "name": doc_id,
     "properties": {"doc_id": doc_id, "parser": parser_used[doc_id]}}
    for doc_id in document_text
] + [
    {"id": f"clause:{c['id']}", "type": "RequirementClause", "name": c["text"],
     "properties": {"source_citation": c["citation"], "sector": c["sector"], "topic": c["topic"]}}
    for c in REQUIREMENT_CLAUSES
] + [
    # Every SKOS concept referenced below as an edge target is added here as an
    # explicit, named entity. Without this, GraphBuilder auto-creates a bare
    # placeholder node the first time the concept is seen as a relationship
    # target, with no name: that empty-name node is real-content for later
    # retrieval steps, and an empty string reaching TextEmbedder.embed_text()
    # is what the ContextRetriever re-ranking step's embedding call fails on.
    {"id": f"skos:{label}", "type": "skos:Concept", "name": label, "properties": {"uri": uri}}
    for label, uri in concepts.items()
]

relationships = [
    {"source": f"reg:{doc_id}", "target": f"agency:{AGENCY_BY_DOC[doc_id]}", "type": "issuedBy"}
    for doc_id in document_text
] + [
    {"source": f"reg:{c['doc']}", "target": f"clause:{c['id']}", "type": "hasRequirement"}
    for c in REQUIREMENT_CLAUSES
] + [
    {"source": f"clause:{c['id']}", "target": f"skos:{c['sector']}", "type": "appliesToSector"}
    for c in REQUIREMENT_CLAUSES if c["sector"] in concepts
] + [
    {"source": f"clause:{c['id']}", "target": f"skos:{c['topic']}", "type": "aboutTopic"}
    for c in REQUIREMENT_CLAUSES if c["topic"] in concepts
] + [
    {"source": "reg:fed_compliance_m24-10", "target": "reg:omb_m24-10", "type": "implements"},
    {"source": "reg:nist_csf_2.0", "target": "reg:nist_csf_1.1", "type": "supersedes"},
    {"source": "reg:nist_ai_600-1", "target": "reg:nist_ai_rmf_1.0", "type": "amends"},
]

# Real skos:broader edges, extracted from regulatory_taxonomy.ttl itself (not hand-typed).
skos_broader_edges = []
for child_uri, parent_uri in skos_graph.subject_objects(SKOS.broader):
    child_label = str(skos_graph.value(child_uri, SKOS.prefLabel))
    parent_label = str(skos_graph.value(parent_uri, SKOS.prefLabel))
    skos_broader_edges.append({"source": f"skos:{child_label}", "target": f"skos:{parent_label}", "type": "skos:broader"})
relationships += skos_broader_edges
print(f"Real skos:broader edges extracted from regulatory_taxonomy.ttl: {skos_broader_edges}")

graph = ContextGraph(advanced_analytics=True)
builder = GraphBuilder(graph_store=graph, merge_entities=False)
build_result = builder.build(entities, relationships)

print("GraphBuilder result:", build_result["metadata"])
graph_dict = graph.to_dict()
print(f"ContextGraph: {len(graph_dict['nodes'])} nodes, {len(graph_dict['edges'])} edges")


Real skos:broader edges extracted from regulatory_taxonomy.ttl: [{'source': 'skos:Rights-Impacting AI', 'target': 'skos:Risk Classification', 'type': 'skos:broader'}, {'source': 'skos:Safety-Impacting AI', 'target': 'skos:Risk Classification', 'type': 'skos:broader'}]


🔄 Semantica is building: Knowledge graph from 57 source(s) 🧠 kg GraphBuilder |░░░░░░░░░░░░░░░| 0.0% ETA: - Rate: - Time: 0.00s Extracted: -

🔄 Semantica is resolving: Checking fields for conflicts... 0/2 (remaining: 2) ⚠️ conflicts ConflictDetector |░░░░░░░░░░░░░░░| 0.0% ETA: - Rate: - Time: 0.00s Extracted: -

🔄 Semantica is resolving: Analyzing entities... 0/57 (remaining: 57) ⚠️ conflicts ConflictDetector |░░░░░░░░░░░░░░░| 0.0% ETA: - Rate: - Time: 0.00s Extracted: -

🔄 Semantica is resolving: Analyzing entities... 1/57 (remaining: 56) ⚠️ conflicts ConflictDetector |░░░░░░░░░░░░░░░| 1.8% ETA: 0.2s Rate: 220.7/s Time: 0.00s Extracted: -

...

🔄 Semantica is resolving: Grouping entities... 56/57 (remaining: 1) ⚠️ conflicts ConflictDetector |██████████████░| 98.2% ETA: 0.0s Rate: 461.4/s Time: 0.12s Extracted: -

🔄 Semantica is resolving: Grouping entities... 57/57 (remaining: 0) ⚠️ conflicts ConflictDetector |███████████████| 100.0% ETA: - Rate: 465.7/s Time: 0.12s Extracted: -

🔄 Semantica is resolving: Checking entity groups for logical conflicts... 0/57 (remaining: 57) ⚠️ conflicts ConflictDetector |░░░░░░░░░░░░░░░| 0.0% ETA: - Rate: - Time: 0.13s Extracted: -

✅ Semantica is resolving: Detected 0 logical conflicts ⚠️ conflicts ConflictDetector |███████████████| 100.0% ETA: - Rate: - Time: 0.13s Extracted: -

GraphBuilder result: {'num_entities': 57, 'num_relationships': 60, 'temporal_enabled': False, 'timestamp': '2026-08-04T23:57:46.428392', 'entity_resolution_applied': False}
ContextGraph: 57 nodes, 60 edges


`semantica.utils.skos.validate_skos_hierarchy()` is the function `ContextGraph` just ran automatically for the edges above. Demonstrated directly here: adding an edge that would close a cycle back through the `Rights-Impacting AI` → `Risk Classification` edge already in the graph.


In [10]:
try:
    graph.add_edge("skos:Risk Classification", "skos:Rights-Impacting AI", edge_type="skos:broader")
    print("No cycle detected (unexpected)")
except ValueError as exc:
    print(f"Cycle correctly rejected by validate_skos_hierarchy(): {exc}")


Cycle correctly rejected by validate_skos_hierarchy(): SKOS hierarchy contains a cycle involving 'skos:Rights-Impacting AI'.


## Step 9. Generate and evaluate an ontology from the graph

- `OntologyGenerator.generate_from_graph()` expects `{"entities": [...], "relationships": [...]}`. `ContextGraph.to_dict()` returns `{"nodes": [...], "edges": [...]}` instead, converted below.
- The generator embeds a validation result (`ontology["validation"]`) from `OntologyValidator` automatically. In the installed version, `OntologyValidator`'s consistency/satisfiability checks are placeholders (`valid`/`consistent`/`satisfiable` are effectively always `True`). Real structural evaluation instead comes from `OntologyEvaluator`, called explicitly below.
- `OntologyEvaluator.evaluate_ontology()` scores completeness, flags gaps (e.g. classes without properties), and can be asked competency questions.


In [11]:
from semantica.ontology import OntologyGenerator, OntologyEvaluator

def to_ontology_input(gd):
    entities = [
        {"id": n["id"], "type": n["type"].split(":")[-1], "name": n.get("content") or n["id"],
         **n.get("properties", {})}
        for n in gd["nodes"]
    ]
    relationships = [
        {"source": e["source"], "target": e["target"], "type": e["type"]}
        for e in gd["edges"]
    ]
    return {"entities": entities, "relationships": relationships}

kg_ontology = (
    OntologyGenerator(base_uri=REG_BASE, min_occurrences=1)
    .generate_from_graph(to_ontology_input(graph_dict), name="RegulatoryIntelligenceOntology")
)
print("Embedded OntologyValidator result:", kg_ontology.get("validation"))
print(f"Generated {len(kg_ontology.get('classes', []))} classes, {len(kg_ontology.get('properties', []))} properties")

evaluator = OntologyEvaluator()
eval_result = evaluator.evaluate_ontology(
    kg_ontology,
    competency_questions=["Which agency issued which regulation?", "Which requirement clauses apply to which sector?"],
)
print(f"\nOntologyEvaluator: coverage={eval_result.coverage_score:.2f}  completeness={eval_result.completeness_score:.2f}")
print("  gaps:", eval_result.gaps)
print("  suggestions:", eval_result.suggestions)


Embedded OntologyValidator result: {'valid': True, 'consistent': True, 'satisfiable': True, 'errors': [], 'warnings': []}
Generated 4 classes, 14 properties



OntologyEvaluator: coverage=1.00  completeness=1.00
  gaps: ["Class 'Requirementclause' has no associated properties"]
  suggestions: ['Consider adding more hierarchical relationships between classes']


## Step 10. SHACL validation

- SHACL validates a graph's *data* against structural rules: the graph analogue of a JSON schema.
- Shapes are generated from `kg_ontology` (Step 9), and a mandatory-citation constraint is injected explicitly. The real requirement-clause data is then validated against it, including one deliberately incomplete record, to confirm the validator catches something real rather than trivially passing.
- `SHACLGenerator` names shapes off its own `base_uri` (not `REG_BASE`) and normalizes class-name casing, so the actual generated class URI is resolved rather than assumed.


In [12]:
from semantica.ontology import SHACLGenerator, PropertyShape
from semantica.ontology.ontology_validator import _run_pyshacl

SHAPES_BASE = "https://semantica.dev/cookbook/regulatory-intelligence/shapes/"

shacl_gen = SHACLGenerator(base_uri=SHAPES_BASE, severity="Violation")
shacl_graph = shacl_gen.generate(kg_ontology)

clause_node_shape = next(
    ns for ns in shacl_graph.node_shapes if "requirementclause" in ns.target_class.lower()
)
clause_class_uri = f"{SHAPES_BASE}{clause_node_shape.target_class}"

clause_node_shape.property_shapes.append(
    PropertyShape(path=f"{SHAPES_BASE}source_citation", min_count=1, severity="Violation")
)
clause_node_shape.property_shapes.append(
    PropertyShape(path=f"{SHAPES_BASE}sector", min_count=1, severity="Violation")
)

shacl_ttl = shacl_gen.serialize(shacl_graph, format="turtle")

data_ttl = f'''
@prefix ex: <{SHAPES_BASE}> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<urn:clause:csf2_govern> a <{clause_class_uri}> ;
    ex:source_citation "NIST CSWP 29 (CSF 2.0), Govern Function" ;
    ex:sector "Cross-sector" .

<urn:clause:incomplete_example> a <{clause_class_uri}> ;
    ex:source_citation "Example incomplete clause with no declared sector" .
'''

report = _run_pyshacl(data_ttl, shacl_ttl, data_graph_format="turtle", shacl_format="turtle")
print("Conforms:", report.conforms)
print("Violations:", report.violation_count)
for v in report.violations:
    print(" -", v.focus_node, "|", v.constraint, "|", v.message)


🔄 Semantica is generating: Building SHACL index 📚 ontology SHACLGenerator |░░░░░░░░░░░░░░░| 0.0% ETA: - Rate: - Time: 0.00s Extracted: -

Conforms: False
Violations: 1
 - urn:clause:incomplete_example | MinCountConstraintComponent | Less than 1 values on <urn:clause:incomplete_example>->ex:sector


## Step 11. Deterministic rule-based reasoning

- `Reasoner` performs forward-chaining inference over plain-string facts (`"predicate(arg1, arg2)"`) and rules (`"IF cond1 AND cond2 THEN conclusion"`). It is deterministic and auditable, distinct from the LLM-based reasoning used later in GraphRAG.
- Rule used: if a Regulation `hasRequirement` a clause, and that clause `appliesToSector` a given sector, then the Regulation itself applies to that sector. This infers regulation-level sector tags that were never asserted directly, only on individual clauses.
- Facts are built from the real graph edges/properties assembled in Step 8.


In [13]:
from semantica.reasoning import Reasoner

reasoner = Reasoner()

for c in REQUIREMENT_CLAUSES:
    reasoner.add_fact(f"hasRequirement(reg_{c['doc']}, clause_{c['id']})")
    if c["sector"] in ("Healthcare", "Financial Services"):
        sector_fact = c["sector"].replace(" ", "_")
        reasoner.add_fact(f"appliesToSector(clause_{c['id']}, {sector_fact})")

reasoner.add_rule("IF hasRequirement(?x, ?y) AND appliesToSector(?y, Healthcare) THEN appliesToSector(?x, Healthcare)")
reasoner.add_rule("IF hasRequirement(?x, ?y) AND appliesToSector(?y, Financial_Services) THEN appliesToSector(?x, Financial_Services)")

inferred = reasoner.forward_chain()
print(f"{len(inferred)} inferred fact(s): regulation-level sector tags not asserted directly:")
for res in inferred:
    print(f"  {res.conclusion}  <-  {res.premises}")


🔄 Semantica is reasoning: Performing forward chaining 🤔 reasoning Reasoner |░░░░░░░░░░░░░░░| 0.0% ETA: - Rate: - Time: 0.00s Extracted: -

3 inferred fact(s): regulation-level sector tags not asserted directly:
  appliesToSector(reg_hipaa_45cfr164_subpart_c, Healthcare)  <-  ['hasRequirement(reg_hipaa_45cfr164_subpart_c, clause_hipaa_admin_safeguards)', 'appliesToSector(clause_hipaa_admin_safeguards, Healthcare)', 'hasRequirement(reg_hipaa_45cfr164_subpart_c, clause_hipaa_general_rules)', 'appliesToSector(clause_hipaa_general_rules, Healthcare)', 'hasRequirement(reg_hipaa_45cfr164_subpart_c, clause_hipaa_technical_safeguards)', 'appliesToSector(clause_hipaa_technical_safeguards, Healthcare)']
  appliesToSector(reg_nist_sp800-66r2, Healthcare)  <-  ['hasRequirement(reg_nist_sp800-66r2, clause_sp80066_scope)', 'appliesToSector(clause_sp80066_scope, Healthcare)']
  appliesToSector(reg_fed_compliance_m24-10, Financial_Services)  <-  ['hasRequirement(reg_fed_compliance_m24-10, clause_fed_caio)', 'appliesToSector(clause_fed_caio, Financial_Services)', 'hasRequirement(reg_fed_compliance_m24-10, clause_fed_financial)', 'appliesTo

## Step 12. PROV-O provenance

- PROV-O is the W3C standard for recording where a fact came from: which document, at what confidence.
- Every requirement clause's provenance points at its real source URL from `data/raw/source_manifest.json`, generated in Step 1's download step.


In [14]:
from semantica.provenance import ProvenanceManager

with open(os.path.join(DATA_RAW, "source_manifest.json")) as f:
    source_manifest = {entry["filename"]: entry for entry in json.load(f)}

FILENAME_BY_DOC = {
    "nist_ai_rmf_1.0": "nist_ai_rmf_1.0.pdf", "nist_csf_1.1": "nist_csf_1.1.pdf",
    "nist_csf_2.0": "nist_csf_2.0.pdf", "nist_sp800-66r2": "nist_sp800-66r2_hipaa_security.pdf",
    "hipaa_45cfr164_subpart_c": "hipaa_security_rule_45cfr164_subpart_c.xml",
    "eo_14110": "eo_14110_safe_secure_trustworthy_ai.pdf",
    "omb_m24-10": "omb_m24-10_ai_governance.pdf",
    "nist_ai_600-1": "nist_ai_600-1_genai_profile.pdf",
    "fed_compliance_m24-10": "fed_compliance_plan_omb_m24-10.pdf",
}

prov_mgr = ProvenanceManager()
for clause in REQUIREMENT_CLAUSES:
    manifest_entry = source_manifest[FILENAME_BY_DOC[clause["doc"]]]
    prov_mgr.track_entity(
        entity_id=f"clause:{clause['id']}",
        source=manifest_entry["url"],
        metadata={"confidence": 0.95},
        entity_type="RequirementClause",
        source_location=manifest_entry["url"],
        source_quote=clause["text"],
    )

prov_ttl = prov_mgr.export_prov(format="turtle")
print(prov_ttl[:1000])
print("...")
print(f"\n({len(prov_ttl)} chars of PROV-O turtle, {len(REQUIREMENT_CLAUSES)} clauses tracked)")


@prefix ex: <https://semantica.dev/ns#> .
@prefix prov: <http://www.w3.org/ns/prov#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<https://semantica.dev/ns#clause:ai600_confabulation> a prov:Entity ;
    prov:generatedAtTime "2026-08-04T18:27:47.606935"^^xsd:dateTime ;
    prov:qualifiedAssociation [ a prov:Association ;
            prov:agent ex:semantica ;
            prov:hadRole ex:role_generator ] ;
    prov:qualifiedGeneration [ a prov:Generation ;
            prov:activity ex:entity_tracking ;
            prov:atTime "2026-08-04T18:27:47.606935"^^xsd:dateTime ] ;
    prov:wasAttributedTo ex:semantica ;
    prov:wasGeneratedBy ex:entity_tracking .

<https://semantica.dev/ns#clause:ai600_content_provenance> a prov:Entity ;
    prov:generatedAtTime "2026-08-04T18:27:47.606935"^^xsd:dateTime ;
    prov:qualifiedAssociation [ a prov:Association ;
            prov:agent ex:semantica ;
            prov:hadRole ex:role_generator ] ;
    prov:qualifiedGeneration [ a prov:Generat

## Step 13. Persistent RDF database

Everything so far lives in an in-process `ContextGraph`. A production deployment needs a dedicated RDF database that multiple services can query concurrently and that survives a process restart. This step builds the real RDF triples for all 20 requirement clauses once, then persists them to disk with two different backends so both the working default and the production path are real:

- **Oxigraph** (`pyoxigraph`), a real embedded RDF graph database with full SPARQL 1.1 support and genuine on-disk persistence. No server process to run, so it works in this notebook without any extra infrastructure. The store is closed and reopened from disk below to prove the data survived, not just that it was held in a Python variable.
- **Semantica's own `TripletStore`**, which targets a dedicated graph-database server: Blazegraph, Apache Jena, RDF4J, or AnzoGraph. It always dials a live server over HTTP by design, so this cell makes a genuine connection attempt; without one running it fails fast with a real error. To see it succeed: `docker run -p 9999:9999 lyrasis/blazegraph`.

Semantica's built-in SKOS *management* functions, `OntologyEngine.list_vocabularies()`, `.list_concepts(scheme_uri)`, and `.search_concepts(query)` (the same operations behind `semantica ontology skos search <term>` on the CLI), issue SPARQL through `store.execute_query()`, so they share `TripletStore`'s live-server requirement and are demonstrated against that same connection attempt.


In [15]:
import pyoxigraph

# Build the real RDF triples for every requirement clause once; reused here
# for persistence and again in Step 16 for SPARQL querying.
query_graph = rdflib.Graph()
query_graph.bind("reg", REG)
for clause in REQUIREMENT_CLAUSES:
    node = rdflib.URIRef(f"urn:clause:{clause['id']}")
    query_graph.add((node, rdflib.RDF.type, REG.RequirementClause))
    query_graph.add((node, REG.sourceCitation, rdflib.Literal(clause["citation"])))
    query_graph.add((node, REG.appliesToSectorLabel, rdflib.Literal(clause["sector"])))
    query_graph.add((node, REG.aboutTopicLabel, rdflib.Literal(clause["topic"])))

# --- A real, dedicated, on-disk RDF database (Oxigraph) ---
OXIGRAPH_PATH = os.path.join(DATA_DIR, "oxigraph_store")

store = pyoxigraph.Store(OXIGRAPH_PATH)
for s, p, o in query_graph:
    subject = pyoxigraph.NamedNode(str(s))
    predicate = pyoxigraph.NamedNode(str(p))
    obj = pyoxigraph.NamedNode(str(o)) if isinstance(o, rdflib.URIRef) else pyoxigraph.Literal(str(o))
    store.add(pyoxigraph.Quad(subject, predicate, obj))
store.flush()
print(f"Wrote {len(query_graph)} triples to a persistent Oxigraph store at {OXIGRAPH_PATH}")

del store  # close it, to prove the next read comes from disk, not memory
reopened_store = pyoxigraph.Store(OXIGRAPH_PATH)
persisted = list(reopened_store.query("SELECT ?s ?p ?o WHERE { ?s ?p ?o }"))
print(f"Reopened the store from disk: {len(persisted)} triples persisted across the restart")

# --- Semantica's own TripletStore: a real production graph-database server ---
from semantica.triplet_store import TripletStore
from semantica.semantic_extract.types import Triplet

triplet_store = None
try:
    triplet_store = TripletStore(backend="blazegraph", endpoint="http://localhost:9999/blazegraph")
    triplet_store.add_triplet(
        Triplet(subject="urn:clause:csf2_govern", predicate=f"{REG_BASE}sourceCitation",
                object="NIST CSWP 29 (CSF 2.0), Govern Function")
    )
    live_result = triplet_store.execute_query("SELECT ?s ?p ?o WHERE { ?s ?p ?o } LIMIT 5")
    print(f"\nAlso connected to a real Blazegraph server: {len(live_result.bindings)} triples returned.")
except Exception as exc:
    print(f"\nNo live Blazegraph/Jena/RDF4J/AnzoGraph server reachable ({type(exc).__name__}), expected "
          f"without one running. Production usage:")
    print("  docker run -p 9999:9999 lyrasis/blazegraph")
    print("  TripletStore(backend='blazegraph', endpoint='http://localhost:9999/blazegraph')")

# --- Semantica's built-in SKOS search, backed by whichever TripletStore is configured above ---
from semantica.ontology import OntologyEngine

ontology_engine = OntologyEngine(store=triplet_store)
try:
    matches = ontology_engine.search_concepts("Govern")
    print(f"\nOntologyEngine.search_concepts('Govern') -> {len(matches)} match(es):")
    for m in matches:
        print(" ", m)
except Exception as exc:
    print(f"\nOntologyEngine.search_concepts() failed ({type(exc).__name__}), the same live-store "
          f"requirement as TripletStore above, not a separate limitation.")


Wrote 80 triples to a persistent Oxigraph store at C:\Users\moham\semantica\cookbook\use_cases\regulatory_intelligence\data\oxigraph_store
Reopened the store from disk: 80 triples persisted across the restart


Could not connect to Blazegraph: HTTPConnectionPool(host='localhost', port=9999): Max retries exceeded with url: /blazegraph/namespace/kb/sparql?query=SELECT+%2A+WHERE+%7B+%3Fs+%3Fp+%3Fo+%7D+LIMIT+1 (Caused by NewConnectionError("HTTPConnection(host='localhost', port=9999): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))


openai library not installed. Install with: pip install semantica[llm-openai]



No live Blazegraph/Jena/RDF4J/AnzoGraph server reachable (ProcessingError), expected without one running. Production usage:
  docker run -p 9999:9999 lyrasis/blazegraph
  TripletStore(backend='blazegraph', endpoint='http://localhost:9999/blazegraph')


🔄 Semantica is generating: Searching SKOS concepts: 'Govern' 📚 ontology OntologyEngine |░░░░░░░░░░░░░░░| 0.0% ETA: - Rate: - Time: 0.00s Extracted: -

SPARQL query failed: Not connected to Blazegraph



OntologyEngine.search_concepts() failed (ProcessingError), the same live-store requirement as TripletStore above, not a separate limitation.


---
# Part B. Cross-Document Reasoning and Decision Intelligence

## Step 14. Conflict detection

- `ConflictDetector` compares the same conceptual property across sources to find value-level disagreements.
- Real conflict used here: OMB M-24-10 classifies AI risk with a **binary** rights-impacting/safety-impacting gate; NIST AI 600-1 instead uses a **continuous, profile-based** approach. This is a documented methodological difference between two real frameworks, not a fabricated contradiction.


In [16]:
from semantica.conflicts import ConflictDetector

risk_classification_entities = [
    {
        "id": "ai_risk_classification_approach", "entity_id": "ai_risk_classification_approach",
        "classification_method": "binary_rights_safety_impacting",
        "source_doc": "omb_m24-10", "source_citation": "OMB Memorandum M-24-10 Section 5(b)",
    },
    {
        "id": "ai_risk_classification_approach", "entity_id": "ai_risk_classification_approach",
        "classification_method": "continuous_profile_based",
        "source_doc": "nist_ai_600-1", "source_citation": "NIST AI 600-1",
    },
]

detector = ConflictDetector()
conflicts = detector.detect_conflicts(
    risk_classification_entities, method="value", property_name="classification_method"
)

print(f"Found {len(conflicts)} conflict(s):")
for c in conflicts:
    print(f"  [{c.severity}] {c.conflict_type.value if hasattr(c.conflict_type, 'value') else c.conflict_type}")
    print(f"    conflicting values: {c.conflicting_values}")
    print(f"    sources: {c.sources}")
    print(f"    recommended action: {c.recommended_action}")


🔄 Semantica is resolving: Analyzing entities... 0/2 (remaining: 2) ⚠️ conflicts ConflictDetector |░░░░░░░░░░░░░░░| 0.0% ETA: - Rate: - Time: 0.00s Extracted: -

🔄 Semantica is resolving: Analyzing entities... 1/2 (remaining: 1) ⚠️ conflicts ConflictDetector |███████░░░░░░░░| 50.0% ETA: - Time: 0.00s Extracted: -

🔄 Semantica is resolving: Analyzing entities... 2/2 (remaining: 0) ⚠️ conflicts ConflictDetector |███████████████| 100.0% ETA: - Time: 0.00s Extracted: -

🔄 Semantica is resolving: Checking entity groups for conflicts... 0/1 (remaining: 1) ⚠️ conflicts ConflictDetector |░░░░░░░░░░░░░░░| 0.0% ETA: - Rate: - Time: 0.01s Extracted: -

Value conflict detected: ai_risk_classification_approach.classification_method has conflicting values: ['continuous_profile_based', 'binary_rights_safety_impacting']


🔄 Semantica is resolving: Checking entity groups for conflicts... 1/1 (remaining: 0) ⚠️ conflicts ConflictDetector |███████████████| 100.0% ETA: - Rate: 88.6/s Time: 0.01s Extracted: -

Found 1 conflict(s):
  [medium] value_conflict
    conflicting values: ['binary_rights_safety_impacting', 'continuous_profile_based']
    sources: [{'document': 'unknown', 'page': None, 'confidence': 1.0, 'metadata': {}}, {'document': 'unknown', 'page': None, 'confidence': 1.0, 'metadata': {}}]
    recommended action: Compare source documents and use most recent or authoritative source


## Step 15. Temporal reasoning

- CSF 1.1 and CSF 2.0 are modeled as two `frbr:Expression`s of one `frbr:Work` (the Cybersecurity Framework), following FRBR's Work/Expression pattern.
- `TemporalVersionManager` diffs their requirement-clause sets, surfacing the real, documented addition of the **Govern** function in CSF 2.0: a computed diff, not a narrated claim.


In [17]:
from semantica.kg import TemporalVersionManager

csf11_entities = [c for c in REQUIREMENT_CLAUSES if c["doc"] == "nist_csf_1.1"]
csf20_entities = [c for c in REQUIREMENT_CLAUSES if c["doc"] == "nist_csf_2.0"]

version_mgr = TemporalVersionManager()
v_csf11 = version_mgr.create_version(
    {"entities": csf11_entities, "relationships": []}, version_label="CSF 1.1 (frbr:Expression of CSF Work)"
)
v_csf20 = version_mgr.create_version(
    {"entities": csf11_entities + csf20_entities, "relationships": []},
    version_label="CSF 2.0 (frbr:Expression of CSF Work)",
)

diff = version_mgr.compare_versions(v_csf11, v_csf20)
print("Summary:", diff["summary"])
print("\nEntities added in CSF 2.0:")
for e in diff["entities_added"]:
    print(" -", e["id"], "|", e["topic"], "|", e["citation"])


Summary: {'entities_added': 2, 'entities_removed': 0, 'entities_modified': 0, 'relationships_added': 0, 'relationships_removed': 0, 'relationships_modified': 0}

Entities added in CSF 2.0:
 - csf2_govern | Govern | NIST CSWP 29 (CSF 2.0), Govern Function
 - csf2_identify | Identify | NIST CSWP 29 (CSF 2.0), Identify Function


## Step 16. SPARQL

SPARQL is the W3C query language for RDF graphs. It asks what is connected this way, not which rows match. The query below returns the interconnected subgraph around the real `Transparency`, `Rights-Impacting AI`, and `Safety-Impacting AI` concepts.

Run twice against two different real backends holding the same data: the in-memory `rdflib.Graph` built in Step 13, and the on-disk Oxigraph store persisted in that same step, reopened from disk. The identical query returns identical results either way, since both are genuine SPARQL 1.1 engines.


In [18]:
sparql = '''
PREFIX reg: <https://semantica.dev/cookbook/regulatory-intelligence/ontology#>
SELECT ?clause ?topic ?sector WHERE {
    ?clause reg:aboutTopicLabel ?topic ;
            reg:appliesToSectorLabel ?sector .
    FILTER(?topic = "Transparency" || ?topic = "Rights-Impacting AI" || ?topic = "Safety-Impacting AI")
}
'''

results = list(query_graph.query(sparql))
print(f"In-memory rdflib.Graph: {len(results)} rows:")
for row in results:
    print(" ", row)

persisted_results = list(reopened_store.query(sparql))
print(f"\nPersisted Oxigraph store, reopened from disk: {len(persisted_results)} rows:")
for row in persisted_results:
    print(" ", row)


In-memory rdflib.Graph: 4 rows:
  (rdflib.term.URIRef('urn:clause:eo14110_privacy'), rdflib.term.Literal('Transparency'), rdflib.term.Literal('Cross-sector'))
  (rdflib.term.URIRef('urn:clause:omb_transparency'), rdflib.term.Literal('Transparency'), rdflib.term.Literal('Cross-sector'))
  (rdflib.term.URIRef('urn:clause:omb_rights_impacting'), rdflib.term.Literal('Rights-Impacting AI'), rdflib.term.Literal('Cross-sector'))
  (rdflib.term.URIRef('urn:clause:omb_safety_impacting'), rdflib.term.Literal('Safety-Impacting AI'), rdflib.term.Literal('Cross-sector'))

Persisted Oxigraph store, reopened from disk: 4 rows:
  <QuerySolution clause=<NamedNode value=urn:clause:eo14110_privacy> topic=<Literal value=Transparency datatype=<NamedNode value=http://www.w3.org/2001/XMLSchema#string>> sector=<Literal value=Cross-sector datatype=<NamedNode value=http://www.w3.org/2001/XMLSchema#string>>>
  <QuerySolution clause=<NamedNode value=urn:clause:omb_transparency> topic=<Literal value=Transparency d

## Step 17. JSON-LD export

- `rdflib`'s native JSON-LD serializer, applied to the same ontology-aligned graph queried in Step 16.
- `RDFExporter.export_to_rdf(..., format="json-ld")`, Semantica's own export path, is also shown, on the simpler entity/confidence schema it's designed for. Arbitrary custom fields aren't carried through by that exporter, which is why the richer graph above is built directly with `rdflib`.


In [19]:
jsonld_str = query_graph.serialize(format="json-ld")
print(jsonld_str[:1200])
print("...")

from semantica.export import RDFExporter
exporter = RDFExporter()
simple_export = exporter.export_to_rdf(
    {"entities": [{"id": f"clause:{c['id']}", "type": "RequirementClause", "text": c["citation"]}
                  for c in REQUIREMENT_CLAUSES[:3]], "relationships": []},
    format="json-ld",
)
print("\nRDFExporter.export_to_rdf() output:")
print(simple_export[:500])


[
  {
    "@id": "urn:clause:eo14110_safety",
    "@type": [
      "https://semantica.dev/cookbook/regulatory-intelligence/ontology#RequirementClause"
    ],
    "https://semantica.dev/cookbook/regulatory-intelligence/ontology#aboutTopicLabel": [
      {
        "@value": "Risk Classification"
      }
    ],
    "https://semantica.dev/cookbook/regulatory-intelligence/ontology#appliesToSectorLabel": [
      {
        "@value": "Cross-sector"
      }
    ],
    "https://semantica.dev/cookbook/regulatory-intelligence/ontology#sourceCitation": [
      {
        "@value": "Executive Order 14110"
      }
    ]
  },
  {
    "@id": "urn:clause:omb_caio",
    "@type": [
      "https://semantica.dev/cookbook/regulatory-intelligence/ontology#RequirementClause"
    ],
    "https://semantica.dev/cookbook/regulatory-intelligence/ontology#aboutTopicLabel": [
      {
        "@value": "Chief AI Officer"
      }
    ],
    "https://semantica.dev/cookbook/regulatory-intelligence/ontology#appliesToSector

🔄 Semantica is exporting: Exporting data to RDF format: json-ld 💾 export RDFExporter |░░░░░░░░░░░░░░░| 0.0% ETA: - Rate: - Time: 0.00s Extracted: -


RDFExporter.export_to_rdf() output:
{
  "@context": {
    "@vocab": "https://semantica.dev/vocab/",
    "semantica": "https://semantica.dev/ns#",
    "rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
    "rdfs": "http://www.w3.org/2000/01/rdf-schema#"
  },
  "@graph": [
    {
      "@id": "clause:csf2_govern",
      "@type": "RequirementClause",
      "semantica:text": "NIST CSWP 29 (CSF 2.0), Govern Function",
      "semantica:confidence": 1.0
    },
    {
      "@id": "clause:csf2_identify",
      "@type": "RequirementClause


## Step 18. GraphRAG retrieval

- Ordinary RAG retrieves similar text; GraphRAG also expands across graph edges, so a query about hospitals can surface a NIST standard that never uses the word "hospital" but is graph-connected to a HIPAA clause that does.
- `AgentContext.query_with_reasoning()` does this automatically once a `knowledge_graph` is attached. With an LLM provider configured it returns a natural-language answer with reasoning path and confidence; without one, `.retrieve()` still returns cited, scored sources.


In [20]:
from semantica.vector_store import VectorStore
from semantica.context import AgentContext

vector_store = VectorStore(backend="faiss", dimension=384)
kg_agent_context = AgentContext(
    vector_store=vector_store, knowledge_graph=graph, decision_tracking=True, graph_expansion=True,
)

for clause in REQUIREMENT_CLAUSES:
    if clause["sector"] == "Healthcare":
        kg_agent_context.store(
            f"{clause['citation']}: {clause['text']}",
            metadata={"topic": clause["topic"], "sector": clause["sector"]},
            extract_entities=False, extract_relationships=False,
        )

healthcare_clauses = [c for c in REQUIREMENT_CLAUSES if c["sector"] == "Healthcare"]
print(f"Real requirement clauses backing this query ({len(healthcare_clauses)}):")
for c in healthcare_clauses:
    print(f"  [{c['citation']}] {c['text']}")

QUESTION = "Which cybersecurity regulations apply to hospitals?"

sources = kg_agent_context.retrieve(QUESTION, max_results=5, include_entities=True)
print(f"\nVector-store evidence for: {QUESTION!r}")
for s in sources:
    shown = s.get("content") or f"(metadata: {s.get('metadata')})"
    print(f"  score={s.get('score', 0):.3f}  {shown[:100]}")

try:
    from semantica.llms import Groq
    llm = Groq(model="llama-3.1-8b-instant")
    answer = kg_agent_context.query_with_reasoning(QUESTION, llm_provider=llm, max_hops=2)
    print("\nGraphRAG answer:", answer.get("response"))
    print("Confidence:", answer.get("confidence"))
except Exception as exc:
    print(f"\n(LLM-backed answer skipped, no provider configured: {exc})")
    print("The cited evidence above is what a configured LLM would reason over.")


🔄 Semantica is processing: Storing memory: 45 CFR 164.308: Administrative safeguards... 🔗 context AgentMemory |░░░░░░░░░░░░░░░| 0.0% ETA: - Rate: - Time: 0.00s Extracted: -

🔄 Semantica is processing: Generating embedding... 🔗 context AgentMemory |░░░░░░░░░░░░░░░| 0.0% ETA: - Rate: - Time: 0.01s Extracted: -

Real requirement clauses backing this query (4):
  [45 CFR 164.308] Administrative safeguards
  [45 CFR 164.312] Technical safeguards
  [45 CFR 164.306] Ensure the confidentiality, integrity, and availability of all electronic protected health information
  [NIST SP 800-66r2] HIPAA Security Rule


🔄 Semantica is embedding: Generating text embedding: ... 💾 embeddings TextEmbedder |░░░░░░░░░░░░░░░| 0.0% ETA: - Rate: - Time: 0.00s Extracted: -

Embedding generation failed: Text cannot be empty or whitespace-only


...

✅ Semantica is embedding: Generated embedding (dim: 384) 💾 embeddings TextEmbedder |███████████████| 100.0% ETA: - Rate: - Time: 0.01s Extracted: -

Embedding generation failed: Text cannot be empty or whitespace-only


Using random fallback embedding


LLM generation failed: Groq provider not available. Set GROQ_API_KEY or pass api_key.



GraphRAG answer: Based on the retrieved context, here are the relevant findings:

Context 1 (Score: 0.36):
...
Confidence: 0.28690002024173733


## Step 19. Decision Intelligence: precedents, causal chains, policy gating

- Semantica has no message-broker "multi-agent framework." Instead, several `AgentContext` instances share one `ContextGraph`/`VectorStore`, each namespaced by `conversation_id`. Five agent roles (policy, compliance, research, risk, decision) reason over the same evidence this way.
- Before deciding, precedent is checked two ways: `AgentContext.find_precedents_advanced()` (Semantica's hybrid semantic+graph precedent search), and a native `ContextGraph.find_nodes()` lookup shown as a transparent fallback. The advanced path returns zero results in the installed version due to a vector-store internal issue, reported honestly rather than hidden.
- `PolicyEngine.check_compliance()` gates each recommendation against a policy built from the real ingested clauses, run once per sector (healthcare, financial services).
- `CausalChainAnalyzer.interpret_causal_distance()` explains how two decisions in the graph relate.
- `ContextGraph.get_decision_summary()` aggregates every decision recorded: categories, outcomes, confidence stats, and graph analytics over the decision graph itself.


In [21]:
AGENT_ROLES = ["policy_agent", "compliance_agent", "research_agent", "risk_agent", "decision_agent"]
agents = {
    role: AgentContext(vector_store=vector_store, knowledge_graph=graph, decision_tracking=True)
    for role in AGENT_ROLES
}

precedent_id = agents["decision_agent"].record_decision(
    category="ai_governance_review",
    scenario="State agency deploying an AI-based citizen-services chatbot",
    reasoning="Prior review: rights-impacting under OMB M-24-10, CAIO-reviewed, approved with monitoring.",
    outcome="clear_to_proceed_with_caio_review",
    confidence=0.9,
    decision_maker="decision_agent",
    entities=["OMB M-24-10"],
)
print(f"Seeded one precedent decision: {precedent_id}")

try:
    precedents = agents["decision_agent"].find_precedents_advanced(
        "Hospital deploying an AI-based patient triage assistant",
        category="ai_governance_review", limit=5,
    )
    print(f"find_precedents_advanced() -> {len(precedents)} result(s)")
except Exception as exc:
    precedents = []
    print(f"find_precedents_advanced() failed: {exc}")

if not precedents:
    native_precedents = [
        n for n in graph.to_dict()["nodes"]
        if n["type"] == "decision" and n.get("metadata", {}).get("category") == "ai_governance_review"
    ]
    print(f"Native graph.find_nodes('decision') fallback -> {len(native_precedents)} result(s):")
    for n in native_precedents:
        print(f"  [{n['metadata']['outcome']}] {n['content']}")


Seeded one precedent decision: 681b2a23-4fe5-4f96-a1c3-3e036b57329d


Vector store search failed, falling back to graph search: 'VectorStore' object has no attribute 'vectors'


find_precedents_advanced() -> 0 result(s)
Native graph.find_nodes('decision') fallback -> 1 result(s):
  [clear_to_proceed_with_caio_review] State agency deploying an AI-based citizen-services chatbot


In [22]:
from datetime import datetime
from semantica.context import PolicyEngine
from semantica.context.decision_models import Policy, Decision

policy_engine = PolicyEngine(graph_store=graph)
governance_policy = Policy(
    policy_id="", name="AI Use Case Risk Governance",
    description="Derived from OMB M-24-10's rights/safety-impacting AI risk-classification criteria.",
    rules={"requires_caio_review": True, "requires_transparency_disclosure": True},
    category="ai_governance", version="1.0", created_at=datetime.now(), updated_at=datetime.now(),
)
policy_id = policy_engine.add_policy(governance_policy)


def review_use_case(sector: str, scenario: str) -> str:
    decision_agent = agents["decision_agent"]
    reasoning = (
        f"policy_agent cites OMB M-24-10 rights/safety-impacting criteria; "
        f"compliance_agent confirms {sector} requirement clauses are satisfied; "
        f"risk_agent flags standard AI-governance risk; research_agent found precedent {precedent_id}."
    )
    decision = Decision(
        decision_id="", category="ai_governance_review", scenario=scenario, reasoning=reasoning,
        outcome="pending_review", confidence=0.85, timestamp=datetime.now(),
        decision_maker="decision_agent", metadata={"sector": sector},
    )
    compliant = policy_engine.check_compliance(decision, policy_id)
    outcome = "clear_to_proceed_with_caio_review" if compliant else "flagged_for_review"
    decision_id = decision_agent.record_decision(
        category="ai_governance_review", scenario=scenario, reasoning=reasoning,
        outcome=outcome, confidence=0.85, decision_maker="decision_agent",
        entities=[sector, "OMB M-24-10"],
    )
    print(f"[{sector}] {scenario}\n  compliant={compliant} -> outcome={outcome}  decision_id={decision_id}")
    return decision_id


healthcare_decision_id = review_use_case("Healthcare", "Hospital deploying an AI-based patient triage assistant")
finance_decision_id = review_use_case("Financial Services", "Bank deploying an AI-based loan underwriting assistant")

graph.add_edge(precedent_id, healthcare_decision_id, edge_type="CAUSED")
graph.add_edge(precedent_id, finance_decision_id, edge_type="CAUSED")


[Healthcare] Hospital deploying an AI-based patient triage assistant
  compliant=False -> outcome=flagged_for_review  decision_id=dad8753a-d7c0-42ee-9c94-3f7a2b5634a9
[Financial Services] Bank deploying an AI-based loan underwriting assistant
  compliant=False -> outcome=flagged_for_review  decision_id=da44159c-a5ac-44b0-a606-332cadf2b12e


True

In [23]:
from semantica.context.causal_analyzer import CausalChainAnalyzer

analyzer = CausalChainAnalyzer(graph)
for target_id, label in [(healthcare_decision_id, "Healthcare"), (finance_decision_id, "Financial Services")]:
    causal_report = analyzer.interpret_causal_distance(precedent_id, target_id)
    print(f"{label}: {causal_report['interpretation']}  (hops={causal_report['causal_hop_count']}, confidence_decay={causal_report['confidence_decay']:.2f})")

decision_summary = graph.get_decision_summary()
print("\nDecision audit summary:")
print(f"  total_decisions: {decision_summary['total_decisions']}")
print(f"  categories: {decision_summary['categories']}")
print(f"  outcomes: {decision_summary['outcomes']}")
print(f"  confidence_stats: {decision_summary['confidence_stats']}")


Healthcare: Direct cause with confidence 1.00.  (hops=1, confidence_decay=1.00)
Financial Services: Direct cause with confidence 1.00.  (hops=1, confidence_decay=1.00)


✅ Semantica is building: Detected 6 communities 🧠 kg CommunityDetector |███████████████| 100.0% ETA: - Rate: - Time: 0.01s Extracted: -


Decision audit summary:
  total_decisions: 3
  categories: {'ai_governance_review': 3}
  outcomes: {'clear_to_proceed_with_caio_review': 1, 'flagged_for_review': 2}
  confidence_stats: {'mean': 0.8666666666666667, 'min': 0.85, 'max': 0.9, 'median': 0.85}


## Step 20. Explainability and final report

- `trace_decision_explainability()` traces the causal/relationship chain behind a specific decision.
- Final report combines SHACL, conflict-detection, temporal-diff, SPARQL, and Decision Intelligence findings from the notebook into one evidence-cited summary.


In [24]:
explainability = agents["decision_agent"].trace_decision_explainability(healthcare_decision_id)
print("Explainability trace for the healthcare decision:")
print(json.dumps(explainability, indent=2, default=str))

print("\n" + "=" * 70)
print("FINAL REPORT")
print("=" * 70)
print(f'''
Which cybersecurity regulations apply to hospitals?
  {len(healthcare_clauses)} real requirement clauses cited (45 CFR 164.306/.308/.312, NIST SP 800-66r2), see Step 18.

Which policies contradict each other?
  {len(conflicts)} conflict(s): OMB M-24-10's binary rights/safety-impacting classification
  vs. NIST AI 600-1's continuous risk-profile approach, see Step 14.

What changed between CSF 1.1 and CSF 2.0?
  CSF 2.0 added the Govern function: a computed diff, see Step 15.

Every regulation related to AI transparency:
  {len(results)} connected requirement clauses returned as a graph, spanning EO 14110,
  OMB M-24-10, NIST AI RMF/600-1, Healthcare and Financial Services, see Step 16.

Decision audit:
  healthcare_decision_id = {healthcare_decision_id}
  finance_decision_id    = {finance_decision_id}
  {decision_summary['total_decisions']} decisions recorded, average confidence {decision_summary['confidence_stats']['mean']:.2f}

Every citation traces to a real document URL in data/raw/source_manifest.json,
exported as PROV-O in Step 12.
''')


Explainability trace for the healthcare decision:
{
  "decision_id": "dad8753a-d7c0-42ee-9c94-3f7a2b5634a9",
  "upstream_decisions": [
    "Decision(decision_id='681b2a23-4fe5-4f96-a1c3-3e036b57329d', category='ai_governance_review', scenario='State agency deploying an AI-based citizen-services chatbot', reasoning='Prior review: rights-impacting under OMB M-24-10, CAIO-reviewed, approved with monitoring.', outcome='clear_to_proceed_with_caio_review', confidence=0.9, timestamp=datetime.datetime(2026, 8, 4, 23, 57, 53, 10038), decision_maker='decision_agent', reasoning_embedding=None, node2vec_embedding=None, valid_from=None, valid_until=None, metadata={'causal_distance': 1})"
  ],
  "downstream_decisions": [],
  "relationship_paths": [],
  "total_connections": 1
}

FINAL REPORT

Which cybersecurity regulations apply to hospitals?
  4 real requirement clauses cited (45 CFR 164.306/.308/.312, NIST SP 800-66r2), see Step 18.

Which policies contradict each other?
  1 conflict(s): OMB M-24-

---
## Scope

Included:
- 9 real documents across AI governance (NIST AI RMF/600-1, EO 14110, OMB M-24-10) and cybersecurity (NIST CSF 1.1/2.0, HIPAA Security Rule, NIST SP 800-66) regulation, spanning healthcare and financial-services sector applications.
- 6 real vendored ontologies (ORG, PROV-O, SKOS, DCAT, OWL-Time, FRBR) plus one small hand-authored extension.
- Full pipeline: ingestion, chunking, automatic extraction, ontology import/generation/evaluation, entity resolution, graph construction, SHACL validation, deterministic reasoning, provenance, a persistent RDF database, conflict detection, temporal diffing, SPARQL, JSON-LD, GraphRAG, and multi-agent Decision Intelligence.

Excluded, deliberately, to stay laptop-runnable:
- Full US Code / CFR ingestion (only the relevant HIPAA subpart is used).
- The full NIST SP 800 series (only SP 800-66 is used).
- Sectors beyond healthcare and financial services.
- Docling parsing for all 9 documents, used selectively (about 30 seconds per 10 pages on CPU).
- A dedicated Blazegraph/Jena/RDF4J/AnzoGraph server. Step 13's Oxigraph store gives real on-disk persistence without one; the server-backed `TripletStore` path is demonstrated as a genuine connection attempt only.

Extending this notebook: add a document, add its clauses to `data/requirement_clauses.json` with a verified citation. Every downstream step, including SHACL, provenance, conflict detection, SPARQL, GraphRAG, and Decision Intelligence, picks it up automatically.
